In [11]:
import os
from PIL import Image
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from collections import Counter
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_pre
from tensorflow.keras.applications.resnet50 import preprocess_input as res_pre
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mob_pre
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np

In [4]:
train_dir = "Dataset/train"
val_dir = "Dataset/val"

In [17]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    horizontal_flip=True,
    zoom_range=0.2
)
val_datagen = ImageDataGenerator(rescale=1./255)

In [19]:
train_ds = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary"      
)
val_ds = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode="binary"      
)
print("Số lớp:", train_ds.num_classes)


Found 14164 images belonging to 2 classes.
Found 1201 images belonging to 2 classes.
Số lớp: 2


In [7]:
base_model = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

# Đóng băng toàn bộ layer VGG16
for layer in base_model.layers:
    layer.trainable = False


In [ ]:
x = Flatten()(base_model.output)
x = Dense(128, activation='relu')(x)
x = Dropout(0.4)(x)

output = Dense(1, activation='sigmoid')(x)  

model = Model(inputs=base_model.input, outputs=output)


In [15]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',   
    metrics=['accuracy']
)



In [12]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3
)


Epoch 1/3
443/443 ━━━━━━━━━━━━━━━━━━━━ 1233s 3s/step - accuracy: 0.9466 - loss: 0.1332 - val_accuracy: 0.8993 - val_loss: 0.2343
Epoch 2/3
443/443 ━━━━━━━━━━━━━━━━━━━━ 1264s 3s/step - accuracy: 0.9508 - loss: 0.1215 - val_accuracy: 0.8984 - val_loss: 0.2238
Epoch 3/3
443/443 ━━━━━━━━━━━━━━━━━━━━ 1313s 3s/step - accuracy: 0.9560 - loss: 0.1113 - val_accuracy: 0.9126 - val_loss: 0.2137


In [13]:
model.save("vgg16_binary_trash.h5")
print("Đã lưu mô hình!")


Đã lưu mô hình!


In [18]:
# đánh giá mô hình trên tập validation
val_loss, val_accuracy = model.evaluate(val_ds)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

38/38 ━━━━━━━━━━━━━━━━━━━━ 90s 2s/step - accuracy: 0.9126 - loss: 0.2137
Validation Loss: 0.2137
Validation Accuracy: 0.9126
